# 01 — Baseline de IOI

Neste notebook vamos medir se o GPT-2 Small resolve **Indirect Object Identification (IOI)**.

Regra do exemplo: se um nome aparece novamente como sujeito da ação, o outro nome é a resposta esperada.

> Exemplo: “John and Mary ... John gave the bag to” → resposta correta: **Mary**.

A medida central será:

$$\text{logit difference}=\text{logit(nome correto)}-\text{logit(nome incorreto)}$$

Um valor positivo favorece o nome correto.

In [1]:
import torch

from ioi_replication import (
    build_ioi_examples,
    load_gpt2_small,
    logits_to_logit_diff,
    select_device,
    tokenize_answers,
)

device = select_device()
print(f"Dispositivo selecionado: {device}")

C:\Users\Marcos\Documents\IOI Replication\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo selecionado: cuda


In [2]:
model = load_gpt2_small(device)

model_shape = (
    model.cfg.n_layers,
    model.cfg.n_heads,
    model.cfg.d_model,
)
assert model_shape == (12, 12, 768)
print("camadas × heads por camada × dimensões:", model_shape)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   9%|▉         | 13/148 [00:00<00:01, 114.05it/s]

Loading weights:  17%|█▋        | 25/148 [00:00<00:01, 89.11it/s] 

Loading weights:  24%|██▎       | 35/148 [00:00<00:02, 47.93it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 234.93it/s]

camadas × heads por camada × dimensões: (12, 12, 768)


In [3]:
prompts, answers = build_ioi_examples()
tokens = model.to_tokens(prompts, prepend_bos=True).to(device)
answer_tokens = tokenize_answers(model, answers, device)

for index, (prompt, (correct, incorrect)) in enumerate(zip(prompts, answers)):
    print(f"{index}: {prompt!r}")
    print(f"   correto={correct!r} | incorreto={incorrect!r}")

print("\ntokens:", tuple(tokens.shape))
print("answer_tokens:", tuple(answer_tokens.shape))

0: 'When John and Mary went to the shops, John gave the bag to'
   correto=' Mary' | incorreto=' John'
1: 'When John and Mary went to the shops, Mary gave the bag to'
   correto=' John' | incorreto=' Mary'
2: 'When Tom and James went to the park, James gave the ball to'
   correto=' Tom' | incorreto=' James'
3: 'When Tom and James went to the park, Tom gave the ball to'
   correto=' James' | incorreto=' Tom'
4: 'When Dan and Sid went to the shops, Sid gave an apple to'
   correto=' Dan' | incorreto=' Sid'
5: 'When Dan and Sid went to the shops, Dan gave an apple to'
   correto=' Sid' | incorreto=' Dan'
6: 'After Martin and Amy went to the park, Amy gave a drink to'
   correto=' Martin' | incorreto=' Amy'
7: 'After Martin and Amy went to the park, Martin gave a drink to'
   correto=' Amy' | incorreto=' Martin'

tokens: (8, 15)
answer_tokens: (8, 2)


## O que o modelo devolve

Os logits terão três eixos:

$$\text{logits.shape}=(\text{batch},\text{sequence},\text{vocabulary})$$

- **batch**: os 8 prompts;
- **sequence**: as posições dos tokens de cada prompt;
- **vocabulary**: uma pontuação para cada token possível.

Como queremos prever o token seguinte, usaremos apenas a última posição da sequência.

In [4]:
with torch.inference_mode():
    logits, cache = model.run_with_cache(tokens)

print("logits:", tuple(logits.shape))
print("última posição:", tuple(logits[:, -1, :].shape))

logits: (8, 15, 50257)
última posição: (8, 50257)


In [5]:
per_prompt = logits_to_logit_diff(
    logits,
    answer_tokens,
    per_prompt=True,
)
average = logits_to_logit_diff(logits, answer_tokens)

for index, difference in enumerate(per_prompt.tolist()):
    favored = "correto" if difference > 0 else "incorreto"
    print(f"prompt {index}: {difference:+.3f} → favorece o nome {favored}")

print(f"\nmédia: {average.item():+.3f}")

prompt 0: +3.337 → favorece o nome correto
prompt 1: +3.202 → favorece o nome correto
prompt 2: +2.709 → favorece o nome correto
prompt 3: +3.797 → favorece o nome correto
prompt 4: +1.720 → favorece o nome correto
prompt 5: +5.281 → favorece o nome correto
prompt 6: +2.601 → favorece o nome correto
prompt 7: +5.767 → favorece o nome correto

média: +3.552


In [6]:
assert logits.shape[:2] == tokens.shape
assert answer_tokens.shape == (8, 2)

prompt_indices = torch.arange(len(prompts), device=device)
manual = (
    logits[prompt_indices, -1, answer_tokens[:, 0]]
    - logits[prompt_indices, -1, answer_tokens[:, 1]]
)

torch.testing.assert_close(per_prompt, manual)
assert average.item() > 0
print("Verificações concluídas: a implementação e o cálculo manual concordam.")

Verificações concluídas: a implementação e o cálculo manual concordam.


## Interpretação

Uma diferença média positiva mostra que, neste conjunto, o modelo tende a favorecer o nome correto.

Isso mede o **comportamento final**, mas ainda não revela quais heads causam esse resultado. Nos próximos notebooks, vamos decompor contribuições e fazer intervenções causais.

Também é importante não confundir a média com acerto em todos os prompts: os valores individuais acima mostram o comportamento caso a caso.